In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)

### Data

#### Ground truth

In [2]:
N = 300_000
issuers     = np.array(['x', 'y', 'z', 'e'])          # one offer per issuer
gamma_true  = np.array([0.090, 0.055, 0.030, 0.015])  # attractiveness = CTR at pos 1
theta_true  = np.array([1.00, 0.55, 0.30])            # examination, normalized th_1 = 1
share       = np.array([0.30, 0.30, 0.25, 0.15])      # traffic share per issuer

# Ranker-induced confounding: better issuers are shown higher (but every
# issuer appears at every position -> the offer-position graph is connected)
P_pos = np.array([[0.70, 0.20, 0.10],   # x mostly slot 1
                  [0.20, 0.50, 0.30],   # y mostly slot 2
                  [0.08, 0.25, 0.67],   # z mostly slot 3
                  [0.05, 0.15, 0.80]])  # e mostly slot 3

#### Simulate rows

In [4]:
iss_idx = rng.choice(4, size=N, p=share)
u       = rng.random(N)
pos     = (u[:, None] > P_pos.cumsum(axis=1)[iss_idx]).sum(axis=1)   # 0,1,2
p_click = theta_true[pos] * gamma_true[iss_idx]
click   = (rng.random(N) < p_click).astype(int)

df = pd.DataFrame({'rank_pos': pos + 1, 'click': click, 'issuer': issuers[iss_idx]})
df.head()

,rank_pos,click,issuer
0,2,0,z
1,2,0,z
2,1,0,e
3,3,0,z
4,3,0,e


#### Naive group-bys

In [5]:
naive_pos = df.groupby('rank_pos')['click'].mean()
naive_iss = df.groupby('issuer')['click'].mean().reindex(issuers)
print("\nNAIVE CTR by position:\n", naive_pos.round(4).to_string())
print("naive ratios th_p/th_1:", (naive_pos / naive_pos.loc[1]).round(3).values,
      " <- truth:", theta_true)
print("\nNAIVE CTR by issuer:\n", naive_iss.round(4).to_string())
print("naive ratios g_o/g_x:", (naive_iss / naive_iss.loc['x']).round(3).values,
      " <- truth:", (gamma_true / gamma_true[0]).round(3))



NAIVE CTR by position:
 rank_pos
1    0.0783
2    0.0295
3    0.0106
naive ratios th_p/th_1: [1.    0.377 0.136]  <- truth: [1.   0.55 0.3 ]

NAIVE CTR by issuer:
 issuer
x    0.0774
y    0.0303
z    0.0129
e    0.0052
naive ratios g_o/g_x: [1.    0.391 0.167 0.067]  <- truth: [1.    0.611 0.333 0.167]


#### Sufficient stats

In [6]:
cells = (df.groupby(['issuer', 'rank_pos'])
           .agg(clicks=('click', 'sum'), imps=('click', 'size'))
           .reset_index())
C  = cells.pivot(index='issuer', columns='rank_pos', values='clicks') \
          .loc[issuers].values.astype(float)
Nm = cells.pivot(index='issuer', columns='rank_pos', values='imps') \
          .loc[issuers].values.astype(float)
print("\nimpressions N_op:\n", Nm.astype(int))
print("clicks C_op:\n", C.astype(int))


impressions N_op:
 [[63212 17850  8991]
 [17997 44976 26920]
 [ 5947 18659 50361]
 [ 2158  6591 36338]]
clicks C_op:
 [[5803  913  256]
 [ 985 1317  418]
 [ 179  312  477]
 [  26   58  151]]


**Assumption**

$$\mathrm{CTR}_{op} = \theta_p \gamma_o \tag{1}$$

where $\theta_p$ is the examination probability of slot $p$ and $\gamma_o$ is the attractiveness of the offer. From (1):

$$\log \mathrm{CTR}_{op} = \log \theta_p + \log \gamma_o$$

$C_{op} \sim \mathrm{Binomial}(N_{op},\, \theta_p \gamma_o)$, and for small $\pi$ and large $N$, $\mathrm{Binomial}(N, \pi) \approx \mathrm{Poisson}(N\pi)$.

**MLE**

$$\mu_{op} = N_{op}\,\theta_p\,\gamma_o$$

$$P(C_{op} = c) = \frac{\mu_{op}^{\,c}\, e^{-\mu_{op}}}{c!} \quad \text{(Poisson)}$$

$$\log L = \sum_{o,p} \bigl[\, C_{op} \log \mu_{op} - \mu_{op} - \log(C_{op}!) \,\bigr]$$

$$\log L = \sum_{o,p} \bigl[\, C_{op} \log(\theta_p \gamma_o) - N_{op}\theta_p \gamma_o \,\bigr] + \sum_{o,p} \bigl[\, C_{op} \log N_{op} - \log(C_{op}!) \,\bigr]$$

$$\log L \;\simeq\; \sum_{o,p} \bigl[\, C_{op} \log(\theta_p \gamma_o) - N_{op}\theta_p \gamma_o \,\bigr] + \text{const}$$

**Notation**

- $C_{o\bullet} = \sum_p C_{op}$ — the row total: all clicks offer $o$ ever got, summed across positions.
- $C_{\bullet p} = \sum_o C_{op}$ — the column total: all clicks at position $p$, summed across offers.

$$\ell = \sum_o \sum_p \bigl[\, C_{op}\log\theta_p + C_{op}\log\gamma_o - N_{op}\theta_p\gamma_o \,\bigr]$$

**Deriving the offer update**

$\partial\ell/\partial\gamma_o$ means: pick one offer, say $o = x$, treat $\gamma_x$ as the variable and everything else (all $\theta_p$, all other $\gamma$'s, all $C$'s and $N$'s) as constants.

$$\frac{\partial \ell}{\partial \gamma_x} = \sum_p \frac{\partial}{\partial \gamma_x}\bigl[\, C_{xp}\log\theta_p + C_{xp}\log\gamma_x - N_{xp}\theta_p\gamma_x \,\bigr]$$

$$\frac{\partial \ell}{\partial \gamma_x} = \sum_p \left[\, \frac{C_{xp}}{\gamma_x} - N_{xp}\theta_p \,\right]$$

$$\sum_p \frac{C_{xp}}{\gamma_x} = \frac{1}{\gamma_x}\sum_p C_{xp} = \frac{C_{x\bullet}}{\gamma_x}$$

$$\frac{\partial \ell}{\partial \gamma_x} = \frac{C_{x\bullet}}{\gamma_x} - \sum_p N_{xp}\theta_p$$

Setting to zero and solving:

$$\boxed{\;\gamma_x = \frac{C_{x\bullet}}{\sum_p N_{xp}\theta_p}\;}$$

By exactly the same reasoning:

$$\boxed{\;\theta_p = \frac{C_{\bullet p}}{\sum_o N_{op}\gamma_o}\;}$$

**Algorithm**

```python
C = cells.pivot(index="offer", columns="pos", values="clicks").fillna(0).values
N = cells.pivot(index="offer", columns="pos", values="imps").fillna(0).values
theta = np.ones(C.shape[1])
for _ in range(500):
    gamma = C.sum(1) / (N @ theta)
    theta = C.sum(0) / (N.T @ gamma)
# since full prob is gamma * thate and for normalizeing we did theta /= theta[0], 
# so for gamma we should do gamma *= theta[0]
gamma *= theta[0]; theta /= theta[0] 
```

**Convergence:** each update exactly maximizes $\ell$ over its block, so $\ell$ never decreases; combined with concavity of $\ell$, the iteration converges to the global optimum.

**ALGORITHM IN DETAILS** `PositionOfferDecomposition`

**Input:** click log rows (offer $o$, position $p$, click $\in \{0,1\}$)
**Output:** $\theta[1..P]$ examination weights with $\theta[1] = 1$; $\gamma[1..O]$ attractiveness (= fitted CTR at position 1)

**Step 0 — collapse to sufficient statistics (the group-by)**

For each observed cell $(o, p)$:

- $N[o,p] \leftarrow$ number of impressions
- $C[o,p] \leftarrow$ number of clicks

$$\mathrm{Crow}[o] \leftarrow \sum_p C[o,p] \quad \text{(total clicks per offer)}$$
$$\mathrm{Ccol}[p] \leftarrow \sum_o C[o,p] \quad \text{(total clicks per position)}$$

*Check:* the bipartite graph on cells with $N[o,p] > 0$ is connected — otherwise effects are identified only within each component.

**Step 1 — initialize**

$$\theta[p] \leftarrow 1 \quad \text{for all } p$$

**Step 2 — alternate exact block maximizations of the Poisson $\ell$**

repeat

&nbsp;&nbsp;&nbsp;&nbsp;for each offer $o$ ($\gamma$-block):
$$\gamma[o] \leftarrow \frac{\mathrm{Crow}[o]}{\sum_p N[o,p]\,\theta[p]} \quad \text{(clicks / "examined" impressions)}$$

&nbsp;&nbsp;&nbsp;&nbsp;for each position $p$ ($\theta$-block):
$$\theta[p] \leftarrow \frac{\mathrm{Ccol}[p]}{\sum_o N[o,p]\,\gamma[o]} \quad \text{(clicks / quality-weighted impressions)}$$

until $\max_p \bigl| \theta_{\text{new}}[p] - \theta_{\text{old}}[p] \bigr| < \mathrm{tol}$

**Step 3 — slide to the $\theta[1]=1$ point on the ridge (both halves!)**

$$c \leftarrow \theta[1], \qquad \theta \leftarrow \theta / c, \qquad \gamma \leftarrow \gamma \cdot c$$

This keeps every product $\theta_p \gamma_o$ unchanged.

**Return** $\theta, \gamma$.

#### The Algorithm

In [7]:
def fit_alternating(C, Nm, tol=1e-12, max_iter=1000, trace=False):
    theta = np.ones(C.shape[1])
    hist  = []
    for it in range(max_iter):
        gamma     = C.sum(axis=1) / (Nm @ theta)        # offer block, closed form
        theta_new = C.sum(axis=0) / (Nm.T @ gamma)      # position block, closed form
        hist.append(theta_new / theta_new[0])
        if np.max(np.abs(theta_new - theta)) < tol:
            theta = theta_new
            break
        theta = theta_new
    gamma = C.sum(axis=1) / (Nm @ theta)
    return theta / theta[0], gamma * theta[0], hist     # normalize th_1 = 1

In [8]:
theta_hat, gamma_hat, hist = fit_alternating(C, Nm, trace=True)

print("\nITERATION TRACE (normalized theta):")
for it in [0, 1, 2, 3, 4, 9, len(hist) - 1]:
    if it < len(hist):
        print(f"  iter {it+1:3d}: theta = {np.round(hist[it], 5)}")
print(f"converged in {len(hist)} iterations")

print("\nALTERNATING estimates:")
print("theta_hat:", theta_hat.round(4), " truth:", theta_true)
print("gamma_hat:", gamma_hat.round(4), " truth:", gamma_true)

# fixed-point / moment-matching check: fitted totals == observed totals
fitted = Nm * gamma_hat[:, None] * theta_hat[None, :]
print("\nmoment check  max|fitted-observed| row totals:",
      f"{np.max(np.abs(fitted.sum(1) - C.sum(1))):.2e}",
      " col totals:", f"{np.max(np.abs(fitted.sum(0) - C.sum(0))):.2e}")


ITERATION TRACE (normalized theta):
  iter   1: theta = [1.      0.68081 0.4378 ]
  iter   2: theta = [1.      0.5913  0.33978]
  iter   3: theta = [1.      0.56417 0.3129 ]
  iter   4: theta = [1.      0.55557 0.30459]
  iter   5: theta = [1.      0.55279 0.30193]
  iter  10: theta = [1.      0.55146 0.30065]
  iter  25: theta = [1.      0.55146 0.30065]
converged in 25 iterations

ALTERNATING estimates:
theta_hat: [1.     0.5515 0.3006]  truth: [1.   0.55 0.3 ]
gamma_hat: [0.092  0.0534 0.0308 0.0141]  truth: [0.09  0.055 0.03  0.015]

moment check  max|fitted-observed| row totals: 4.55e-13  col totals: 9.19e-10


#### Bootstrap CIs

In [19]:
# rows are iid here, so bootstrapping rows == multinomial resampling of the
# 24 categories (12 cells x {click, no click}); with real widget data,
# resample impression ids instead.
p24  = np.concatenate([C.ravel(), (Nm - C).ravel()]) / N
reps = 1000
boot = np.empty((reps, 5))                     # th2, th3, gy/gx, gz/gx, ge/gx
for b in range(reps):
    cnt = rng.multinomial(N, p24)
    Cb  = cnt[:12].reshape(4, 3).astype(float)
    Nb  = Cb + cnt[12:].reshape(4, 3)
    th_b, ga_b, _ = fit_alternating(Cb, Nb)
    boot[b] = [th_b[1], th_b[2], ga_b[1]/ga_b[0], ga_b[2]/ga_b[0], ga_b[3]/ga_b[0]]

lo, hi = np.percentile(boot, [2.5, 97.5], axis=0)
names  = ['theta_2/theta_1', 'theta_3/theta_1',
          'gamma_y/gamma_x', 'gamma_z/gamma_x', 'gamma_e/gamma_x']
truthv = [theta_true[1], theta_true[2],
          gamma_true[1]/gamma_true[0], gamma_true[2]/gamma_true[0], gamma_true[3]/gamma_true[0]]
est    = [theta_hat[1], theta_hat[2],
          gamma_hat[1]/gamma_hat[0], gamma_hat[2]/gamma_hat[0], gamma_hat[3]/gamma_hat[0]]
print("\nBOOTSTRAP 95% CIs (1000 reps):")
for n, e, l, h, t in zip(names, est, lo, hi, truthv):
    print(f"  {n}: {e:.3f}  [{l:.3f}, {h:.3f}]   truth {t:.3f}")



BOOTSTRAP 95% CIs (1000 reps):
  theta_2/theta_1: 0.551  [0.525, 0.580]   truth 0.550
  theta_3/theta_1: 0.301  [0.280, 0.322]   truth 0.300
  gamma_y/gamma_x: 0.581  [0.552, 0.611]   truth 0.611
  gamma_z/gamma_x: 0.335  [0.310, 0.361]   truth 0.333
  gamma_e/gamma_x: 0.153  [0.132, 0.175]   truth 0.167
